In [2]:
# ============================================================
# CELL 1: Import Library
# ============================================================
import numpy as np
import pandas as pd
import pickle
import json
import random
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("✅ Library loaded")
print(f"✅ Seed = {SEED}")
print(f"✅ TensorFlow version: {tf.__version__}")

✅ Library loaded
✅ Seed = 42
✅ TensorFlow version: 2.21.0


In [3]:
# ============================================================
# CELL 2: Load Dataset
# ============================================================
df = pd.read_csv('dataset/Crop_recommendation.csv')

print(f"✅ Dataset loaded")
print(f"   Shape  : {df.shape}")
print(f"   Kolom  : {list(df.columns)}")
print(f"   Kelas  : {df['label'].nunique()}")
print(f"   Missing: {df.isnull().sum().sum()}")
df.head()

✅ Dataset loaded
   Shape  : (2200, 8)
   Kolom  : ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall', 'label']
   Kelas  : 22
   Missing: 0


,N,P,K,temperature,humidity,ph,rainfall,label
0,90,42,43,20.879744,82.002744,6.502985,202.935536,rice
1,85,58,41,21.770462,80.319644,7.038096,226.655537,rice
2,60,55,44,23.004459,82.320763,7.840207,263.964248,rice
3,74,35,40,26.491096,80.158363,6.980401,242.864034,rice
4,78,42,42,20.130175,81.604873,7.628473,262.717340,rice


In [4]:
# ============================================================
# CELL 3: Label Encoding
# Rujukan: BAB 3 hal 43, Tabel 3.7
# ============================================================
le = LabelEncoder()
y  = le.fit_transform(df['label'])
X  = df[['N','P','K','temperature',
          'humidity','ph','rainfall']].values

print("✅ Label Encoding selesai")
print(f"   Jumlah kelas : {len(le.classes_)}")
print(f"   Shape X      : {X.shape}")
print(f"   Shape y      : {y.shape}")
print(f"\n   Mapping lengkap:")
for i, crop in enumerate(le.classes_):
    print(f"   {i:2d} → {crop}")

✅ Label Encoding selesai
   Jumlah kelas : 22
   Shape X      : (2200, 7)
   Shape y      : (2200,)

   Mapping lengkap:
    0 → apple
    1 → banana
    2 → blackgram
    3 → chickpea
    4 → coconut
    5 → coffee
    6 → cotton
    7 → grapes
    8 → jute
    9 → kidneybeans
   10 → lentil
   11 → maize
   12 → mango
   13 → mothbeans
   14 → mungbean
   15 → muskmelon
   16 → orange
   17 → papaya
   18 → pigeonpeas
   19 → pomegranate
   20 → rice
   21 → watermelon


In [5]:
# ============================================================
# CELL 4: Split Data 70-15-15
# Rujukan: BAB 3 hal 45-46
# ============================================================
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=SEED,
    stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=SEED,
    stratify=y_temp
)

print("✅ Split Data selesai")
print(f"   Train : {X_train.shape[0]} sampel (70%)")
print(f"   Val   : {X_val.shape[0]} sampel (15%)")
print(f"   Test  : {X_test.shape[0]} sampel (15%)")

# Verifikasi distribusi per kelas
print(f"\n   Verifikasi distribusi (5 kelas pertama):")
for i in range(5):
    n_train = np.sum(y_train == i)
    n_val   = np.sum(y_val   == i)
    n_test  = np.sum(y_test  == i)
    print(f"   {le.classes_[i]:15}: "
          f"train={n_train}, val={n_val}, test={n_test}")

✅ Split Data selesai
   Train : 1540 sampel (70%)
   Val   : 330 sampel (15%)
   Test  : 330 sampel (15%)

   Verifikasi distribusi (5 kelas pertama):
   apple          : train=70, val=15, test=15
   banana         : train=70, val=15, test=15
   blackgram      : train=70, val=15, test=15
   chickpea       : train=70, val=15, test=15
   coconut        : train=70, val=15, test=15


In [11]:
# ============================================================
# CELL 5: StandardScaler
# Rujukan: BAB 3 hal 44-45, Persamaan 3.3
# PENTING: fit HANYA pada train — hindari data leakage!
# ============================================================
scaler         = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

print("✅ StandardScaler selesai")

✅ StandardScaler selesai


In [12]:
# ============================================================
# CELL 5b: Verifikasi dan tampilkan contoh hasil scaling
# ============================================================
print("✅ Hasil StandardScaler:")
print(f"   X_train: mean={X_train_scaled.mean():.4f}, std={X_train_scaled.std():.4f}")
print(f"   X_val  : mean={X_val_scaled.mean():.4f},  std={X_val_scaled.std():.4f}")
print(f"   X_test : mean={X_test_scaled.mean():.4f},  std={X_test_scaled.std():.4f}")

# Ambil 1 sampel pertama dari training set (sebelum dan sesudah scaling)
feature_names = ['N', 'P', 'K', 'Temperature', 'Humidity', 'pH', 'Rainfall']

sample_raw    = X_train[0]      # nilai asli
sample_scaled = X_train_scaled[0]  # nilai setelah scaling

print("\nContoh hasil transformasi (sampel pertama training set):")
df_example = pd.DataFrame({
    'Fitur'    : feature_names,
    'X_raw'    : sample_raw.round(4),
    'X_scaled' : sample_scaled.round(4)
})
print(df_example.to_string(index=False))

✅ Hasil StandardScaler:
   X_train: mean=-0.0000, std=1.0000
   X_val  : mean=0.0010,  std=0.9903
   X_test : mean=0.0134,  std=0.9949

Contoh hasil transformasi (sampel pertama training set):
      Fitur   X_raw  X_scaled
          N  1.0000   -1.3359
          P 67.0000    0.4175
          K 21.0000   -0.5351
Temperature 27.5214    0.3783
   Humidity 60.5366   -0.4894
         pH  6.5516    0.1055
   Rainfall 48.0649   -1.0061


In [7]:
# ============================================================
# CELL 6: One-Hot Encoding Label
# ============================================================
num_classes = len(le.classes_)
y_train_oh  = tf.keras.utils.to_categorical(y_train, num_classes)
y_val_oh    = tf.keras.utils.to_categorical(y_val,   num_classes)
y_test_oh   = tf.keras.utils.to_categorical(y_test,  num_classes)

print("✅ One-Hot Encoding selesai")
print(f"   y_train: {y_train_oh.shape}")
print(f"   y_val  : {y_val_oh.shape}")
print(f"   y_test : {y_test_oh.shape}")

# Verifikasi
print(f"\n   Contoh y_train[0]:")
print(f"   Label integer : {y_train[0]} ({le.classes_[y_train[0]]})")
print(f"   Label one-hot : {y_train_oh[0]}")

✅ One-Hot Encoding selesai
   y_train: (1540, 22)
   y_val  : (330, 22)
   y_test : (330, 22)

   Contoh y_train[0]:
   Label integer : 10 (lentil)
   Label one-hot : [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [8]:
# ============================================================
# CELL 7: Transformasi ECOCROP
# Hanya temperature dan pH (rainfall dihapus)
# Rujukan: BAB 3 hal 40-42, Persamaan 3.1 dan 3.2
# ============================================================
with open('dataset/constraints_optimal.json', 'r') as f:
    ecocrop_minmax = json.load(f)

# Hanya ambil temperature dan ph
PHYSICS_FEATURES = ['temperature', 'ph']

ecocrop_gaussian = {}
for crop, features in ecocrop_minmax.items():
    ecocrop_gaussian[crop] = {}
    for feat in PHYSICS_FEATURES:
        min_v, max_v = features[feat]
        mu    = (min_v + max_v) / 2
        sigma = (max_v - min_v) / 6
        ecocrop_gaussian[crop][feat] = [round(mu, 4), round(sigma, 4)]

# Verifikasi Rice
print("✅ Transformasi ECOCROP selesai (temp + pH only)")
print(f"\nVerifikasi Rice:")
print(f"  Temp : μ={ecocrop_gaussian['rice']['temperature'][0]}, "
      f"σ={ecocrop_gaussian['rice']['temperature'][1]}")
print(f"  pH   : μ={ecocrop_gaussian['rice']['ph'][0]}, "
      f"σ={ecocrop_gaussian['rice']['ph'][1]}")

print(f"\nSemua 22 tanaman:")
for crop, feats in ecocrop_gaussian.items():
    print(f"  {crop:15} | "
          f"temp: μ={feats['temperature'][0]:6.2f} "
          f"σ={feats['temperature'][1]:5.2f} | "
          f"ph: μ={feats['ph'][0]:5.2f} "
          f"σ={feats['ph'][1]:5.2f}")

✅ Transformasi ECOCROP selesai (temp + pH only)

Verifikasi Rice:
  Temp : μ=25.0, σ=1.6667
  pH   : μ=6.25, σ=0.25

Semua 22 tanaman:
  rice            | temp: μ= 25.00 σ= 1.67 | ph: μ= 6.25 σ= 0.25
  maize           | temp: μ= 25.50 σ= 2.50 | ph: μ= 6.00 σ= 0.33
  chickpea        | temp: μ= 22.00 σ= 2.33 | ph: μ= 7.25 σ= 0.42
  kidneybeans     | temp: μ= 20.50 σ= 1.50 | ph: μ= 6.50 σ= 0.33
  pigeonpeas      | temp: μ= 28.00 σ= 3.33 | ph: μ= 6.00 σ= 0.33
  mothbeans       | temp: μ= 28.00 σ= 1.33 | ph: μ= 7.00 σ= 0.17
  mungbean        | temp: μ= 28.50 σ= 2.50 | ph: μ= 5.85 σ= 0.12
  blackgram       | temp: μ= 28.50 σ= 2.17 | ph: μ= 6.00 σ= 0.17
  lentil          | temp: μ= 22.00 σ= 2.33 | ph: μ= 6.50 σ= 0.33
  pomegranate     | temp: μ= 27.50 σ= 1.50 | ph: μ= 7.00 σ= 0.17
  banana          | temp: μ= 27.00 σ= 1.67 | ph: μ= 6.00 σ= 0.33
  mango           | temp: μ= 27.00 σ= 1.00 | ph: μ= 6.50 σ= 0.33
  grapes          | temp: μ= 24.00 σ= 2.00 | ph: μ= 6.75 σ= 0.25
  watermelon      | 

In [9]:
# ============================================================
# CELL 8: Simpan Session
# ============================================================
import os
os.makedirs('session', exist_ok=True)

session = {
    # Raw data
    'X_train'           : X_train,
    'X_val'             : X_val,
    'X_test'            : X_test,
    'y_train'           : y_train,
    'y_val'             : y_val,
    'y_test'            : y_test,

    # Scaled data
    'X_train_scaled'    : X_train_scaled,
    'X_val_scaled'      : X_val_scaled,
    'X_test_scaled'     : X_test_scaled,

    # One-hot label
    'y_train_oh'        : y_train_oh,
    'y_val_oh'          : y_val_oh,
    'y_test_oh'         : y_test_oh,

    # Encoder & scaler
    'le'                : le,
    'scaler'            : scaler,
    'num_classes'       : num_classes,

    # ECOCROP (temperature + pH only)
    'ecocrop_minmax'    : ecocrop_minmax,
    'ecocrop_gaussian'  : ecocrop_gaussian,
    'physics_features'  : PHYSICS_FEATURES,  # ← catat fitur yang dipakai

    # Config
    'SEED'              : SEED,
}

with open('session/session_preprocessing.pkl', 'wb') as f:
    pickle.dump(session, f)

print("✅ Session tersimpan!")
print(f"   Physics features: {PHYSICS_FEATURES}")
print(f"   Keys: {list(session.keys())}")

✅ Session tersimpan!
   Physics features: ['temperature', 'ph']
   Keys: ['X_train', 'X_val', 'X_test', 'y_train', 'y_val', 'y_test', 'X_train_scaled', 'X_val_scaled', 'X_test_scaled', 'y_train_oh', 'y_val_oh', 'y_test_oh', 'le', 'scaler', 'num_classes', 'ecocrop_minmax', 'ecocrop_gaussian', 'physics_features', 'SEED']
